## RL-Based Distillation with LLM-as-a-Judge (Label-Free)

### Key Concepts Covered

* **The Limitation of Token-Level KL:** Why optimizing for token-level probabilities fails on open-ended tasks where multiple distinct responses are equally valid.

* **Framework Mechanics:**
  - **Generator (Student):** Generates responses $y \sim q_\theta(\cdot \mid x)$ given prompt $x$.
  - **Judge (Teacher / Stronger LLM):** Evaluates responses $y$ directly at the sequence level, assigning scalar quality scores $R(x, y)$ based on rubrics (e.g., correctness, fluency, safety, helpfulness).

* **Reward Function & KL Penalty:** Optimizing expected reward while maintaining a reference penalty to prevent policy collapse:
  $$\mathcal{J}(\theta) = \mathbb{E}_{(x,y) \sim q_\theta} \left[ R_{\text{Judge}}(x, y) - \beta \, \mathbb{D}_{\text{KL}}\!\left(q_\theta(y \mid x) \;\Vert\; p_{\text{ref}}(y \mid x)\right) \right]$$

* **Label-Free Distillation:** How this setup eliminates the need for expensive ground-truth human annotations by leveraging unlabelled prompt distributions $p_x$.

* **Optimization Mechanisms:** Contrast with standard MiniLLM—using sequence-level preference/judgment signals via policy optimization algorithms (e.g., PPO, DPO, or REINFORCE with baseline) instead of per-token reverse KL.

## Motivation: Beyond Token-Level Likelihood

Standard knowledge distillation methods—including **forward KL** ($\text{KL}[p \parallel q]$) and **MiniLLM's reverse KL** ($\text{KL}[q \parallel p]$)—operate directly at the token level. They force the student to match the teacher's exact vocabulary logits at every step.

This creates fundamental limitations in open-ended language tasks:

### Semantic Equivalence Penalty

If a prompt asks "How do I fix a leaky faucet?", there are dozens of equally valid ways to structure the response. Token-level KL heavily penalizes a student if its phrasing deviates from the teacher's exact word choices, even if the student's solution is semantically superior.

### Distribution Shift on Open-Ended Rollouts

In long-form generation, minor discrepancies early in a response compound over time, dragging token-level likelihood objectives off-track.

### The Need for Sequence-Level Evaluation

Instead of forcing the student to mimic exact probability vectors over every token in the vocabulary $V$, **RL-based distillation** evaluates the student's generation at the sequence level based on holistic quality, accuracy, and task completion.

## Framework & Architecture

The **Label-Free RL-Based Distillation** paradigm bypasses the need for annotated human datasets or exact token-matching targets. It relies on an unlabelled distribution of prompts ($p_x$) and uses a stronger teacher model as an evaluator rather than a target distribution.

### Label-Free RL Distillation Pipeline

```
  Unlabelled Prompts x ~ p_x
             │
             ▼
    ┌─────────────────┐
    │  Student Model  │ ──► Generates Response y ~ q_θ(·|x)
    │   q_θ (Policy)  │
    └─────────────────┘
             │
             ├──────────────────────────────────────┐
             ▼                                      ▼
    ┌─────────────────┐                    ┌──────────────────┐
    │  Judge (LLM)    │                    │ Reference Model  │
    │ Assigns Score R │                    │  p_ref (Frozen)  │
    └─────────────────┘                    └──────────────────┘
             │                                      │
             ▼                                      ▼
       Reward R(x, y)                       KL Penalty D_KL(q_θ || p_ref)
             │                                      │
             └──────────────────┬───────────────────┘
                                ▼
                       Combined Objective:
                   J(θ) = R(x, y) - β · D_KL
                                │
                                ▼
                   Update Student via Policy Gradient
```

## Core Components

* **Student Policy ($q_\theta$):** The lightweight model being distilled. It receives an unlabelled prompt $x$ and generates a complete response $y \sim q_\theta(\cdot \mid x)$.

* **LLM-as-a-Judge ($R_{\text{Judge}}$):** A powerful teacher model (e.g., GPT-4o, Claude 3.5, or a fine-tuned 70B model) that evaluates $y$ given $x$. It returns a scalar reward score $R(x, y) \in [\text{min}, \text{max}]$ based on structured multi-dimensional rubrics (e.g., factual accuracy, conciseness, adherence to constraints, safety).

* **Reference Policy ($p_{\text{ref}}$):** A frozen baseline copy of the student (or initial SFT model) used to prevent the student from drifting into unreadable or degenerate text while maximizing the judge's score.

## Mathematical Formulation

The overall optimization goal is to maximize the expected score assigned by the LLM Judge while penalizing drift from the reference distribution:

$$\mathcal{J}(\theta) = \mathbb{E}_{x \sim p_x, \, y \sim q_\theta(\cdot \mid x)} \left[ R_{\text{Judge}}(x, y) - \beta \, \mathbb{D}_{\text{KL}}\!\left(q_\theta(y \mid x) \;\Vert\; p_{\text{ref}}(y \mid x)\right) \right]$$

### Where:

* $R_{\text{Judge}}(x, y)$ is the sequence-level score produced by prompting the evaluator LLM.

* $\beta$ is the KL-penalty scaling hyperparameter.

* $\mathbb{D}_{\text{KL}}\!\left(q_\theta(y \mid x) \;\Vert\; p_{\text{ref}}(y \mid x)\right) = \sum_{t=1}^T \log \frac{q_\theta(y_t \mid y_{<t}, x)}{p_{\text{ref}}(y_t \mid y_{<t}, x)}$ is calculated token-by-token across the rollout.